In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches


# Model agnostic 
from typing import Optional, List, Callable, Dict, Any, List
from pathlib import Path
from src.utils import Helm, QLatticeWrapper
from src.dfm_src import DFT

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# Get the directory this file lives in
nb_dir = Path.cwd() # notebook directory
project_root = nb_dir.parents[0] # project directory
data_path = project_root / "datasets" / "processed_well_data.csv"

includ_cols = ['Dia', 'Dev(deg)','Area (m2)', 'z','GasDens','LiquidDens', 'g (m/s2)', 'P/T','friction_factor', 'critical_film_thickness']
D = Helm(path=data_path, includ_cols=includ_cols, test_size=0.20, scale=False) # cannot scale input units 

In [3]:
from src.dfm_src import DFT

def build_dfm(hparams):
    return DFT(
        feature_tol=hparams.get("feature_tol", 1.0),
        dev_tol=hparams.get("dev_tol", 1e-3),
        multiple_dev_policy=hparams.get("multiple_dev_policy", "max"),
    )

hparam_grid = {
    "feature_tol":          [0.5, 1.0, 2.0],
    "multiple_dev_policy":  ["max", "mean"],
}

trained_model = D.evolv_dfm_model(
    build_model=build_dfm,
    hparam_grid=hparam_grid,
    k_folds=5,
)


Training DFM model and optimising hyperparameters via k-fold CV...


Optimization terminated successfully.
         Current function value: 0.144107
         Iterations: 3
         Function evaluations: 15789
True
Optimization terminated successfully.
         Current function value: 0.145963
         Iterations: 3
         Function evaluations: 15898
True
Optimization terminated successfully.
         Current function value: 0.141787
         Iterations: 3
         Function evaluations: 15915
True
Optimization terminated successfully.
         Current function value: 0.142957
         Iterations: 3
         Function evaluations: 15886
True
Optimization terminated successfully.
         Current function value: 0.145614
         Iterations: 3
         Function evaluations: 15884
True
Optimization terminated successfully.
         Current function value: 0.144107
         Iterations: 3
         Function evaluations: 15789
True
Optimization terminated successfully.
         Current function value: 0.145963
         Iterations: 3
         Function evaluatio

Retraining optimised DFM model on full training set...


Optimization terminated successfully.
         Current function value: 0.144924
         Iterations: 3
         Function evaluations: 19722
True



Best CV Classification Accuracy: 
>>> 0.6993 ± 0.0740

Best CV Per-Class Accuracy:
>>> Unloaded (0): 0.7273, Near L.U. (1): 0.0000, Loaded (2): 0.7123

Best Hyperparameters:
>>> {'feature_tol': 1.0, 'multiple_dev_policy': 'max'}

Training Classification Accuracy:
>>> 0.9699

Per-Class Training Accuracy:
>>> Unloaded (0): 1.0000, Near L.U. (1): 0.0000, Loaded (2): 1.0000

Test Classification Accuracy:
>>> 0.6905

Per-Class Test Accuracy:
>>> Unloaded (0): 0.7391, Near L.U. (1): 0.0000, Loaded (2): 0.6667


In [4]:
"""
# define xgboost pipeline
def dft(
        hparams: Dict[str, Any]
):
    dft = DFT(
        **hparams,
    )

    return dft

hparam_grid = {
            "dev_tol":  [1e-5], # [1e-5, 1e-4, 1e-2],
            "feature_tol": [1.0], #[0.5, 1.0, 2.0],
        }
hparam_grid["interval"] = [0]

# train model and optimize hyperparameters via grid search 
trained_model = D.evolv_model(build_model=dft, hparam_grid=hparam_grid, k_folds=5)
"""

'\n# define xgboost pipeline\ndef dft(\n        hparams: Dict[str, Any]\n):\n    dft = DFT(\n        **hparams,\n    )\n\n    return dft\n\nhparam_grid = {\n            "dev_tol":  [1e-5], # [1e-5, 1e-4, 1e-2],\n            "feature_tol": [1.0], #[0.5, 1.0, 2.0],\n        }\nhparam_grid["interval"] = [0]\n\n# train model and optimize hyperparameters via grid search \ntrained_model = D.evolv_model(build_model=dft, hparam_grid=hparam_grid, k_folds=5)\n'